# A Review of the `XBRL-US` Python Package
The XBRL-US package has been available since 2023. It was recently updated with many new features that are not available in any version prior to `1.0.0`.

Make sure to install the latest version so you can take full advantage of all the new features.

In [ ]:
print("Please wait while packages and dependencies are installed. \nWhen the XBRL US Web account authentication prompts appear, enter each and press enter to continue.")
%pip install -q --upgrade pip
%pip install -q xbrl_us==1.0.0
import getpass
from xbrl_us import XBRL
xbrl = XBRL(
    username = input('Enter your XBRL US Web account email: '),
    password = getpass.getpass(prompt='Password: '),
    client_id = getpass.getpass(prompt='Client ID: '),
    client_secret = getpass.getpass(prompt='Client secret: '),
    store = 'y' if input("(y/n) - Do you want to store credentials to use xbrl = XBRL() \nwithout logging in next time? ").strip().lower() == 'y' else 'n'
)
print("This account is ready to use the Python package to query the \nXBRL US Database of Public Filings with the XBRL API.")

In [ ]:
# A simple query to get the last 20 reports
report_response = xbrl.query( 
    endpoint="/report/search",
    fields=[
        "report_id",
        "report_entity_name",
        "report_filing_date",
        "report_base_taxonomy",
        "report_document_type",
        "report_accession",
        "entity_ticker",
        "report_sic_code",
        "entity_cik",
        "report_entry_type",
        "report_period_end",
        "report_sec_url",
        "report_checks_run",
        "report_accepted_timestamp",
    ],
    sort={"report_accepted_timestamp": "desc"},
    limit=100,
    as_dataframe=True,
)

report_response.head(5)

# Endpoint-Specific Methods

Each endpoint has its own query method that sets the allowed endpoints, fields, parameters, and sorting options. For example, use the query method `xbrl.report()` for the report endpoint.

The example below works the same as the query above but uses autocompletion to show only the allowed endpoints, fields, and parameters for report queries.

<div class="alert alert-block alert-info">
  <strong>Note:</strong> The <code>xblr_us</code> Python package supports snake_case names (names with underscores). For example, <code>report_id</code> is the same as <code>report.id</code> and <code>report_entity_name</code> is the same as <code>report.entity-name</code>.
</div>

In [ ]:
# the same query using the report method
report_response = xbrl.report( 
    endpoint="/report/search",
    fields=[
        "report_id",
        "report_entity_name",
        "report_filing_date",
        "report_base_taxonomy",
        "report_document_type",
        "report_accession",
        "entity_ticker",
        "report_sic_code",
        "entity_cik",
        "report_entry_type",
        "report_period_end",
        "report_sec_url",
        "report_checks_run",
        "report_accepted_timestamp"
    ],
    sort={"report_accepted_timestamp": "desc"},
    limit=20,
    as_dataframe=True,
)

# as you can see the query keyword arguments are exactly the same for both methods.

report_response.head(10)

Here are some other examples of queries using specific endpoints. For example, the example below uses `xbrl.assertion()` to build a query related to Center for Data Quality Software Certification page.

In [ ]:
# Center for Data Quality Software Certification page query
assertion_response = xbrl.assertion(  # you can use `xbrl.query()` as well.
    endpoint="/assertion/search",
    parameters={
        "assertion_source": "DQC"
    },
    fields=[
        "report.accession",
        "entity.*",
        "report.document-type",
        "report.filing-date",
        "assertion.code",
        "assertion.type",
        "assertion.detail"
    ],
    sort={"report_filing_date": "desc"},
    limit=1,
    as_dataframe=True
)

assertion_response.head()

<div class="alert alert-block alert-info">
  <strong>Note:</strong> When sorting queries, you have two options for including a field:
  <ul>
    <li>
      Include it only in <code>sort</code> – the field will be used solely for sorting.
    </li>
    <li>
      Include it in both <code>fields</code> and <code>sort</code> – the order specified in the <code>fields</code> parameter will determine its position in the output.
    </li>
  </ul>
  If a sort field is not included in <code>fields</code>, it will be appended to the end of the query results.
</div>

In [ ]:
# Line items in Apple's 2018 10-K Cash Flow Statement
cash_flow_items = xbrl.relationship(  # you can use `xbrl.query()` as well.
    endpoint="/relationship/search",
    parameters={
        "dts_id": 306447,
        "network_role_description": "cash flow",
        "network_link_name": "presentationLink",
        "relationship_target_is_abstract": "false",
    },
    fields=[
        "dts_id", # when you include a field both in `fields` and `sort`, the order of the `fields` will be used for the output
        "relationship_id", 
        "network_role_description",
        "relationship_target_name",
        "relationship_source_namespace"
    ],
    sort={
        "dts_id": "asc", 
        "relationship_id": "asc", 
        "relationship_tree_sequence": "asc" # if you do not include a field in the `fields`, the field will be added to the end
    },
    as_dataframe=True
)

cash_flow_items.head(10)

In [ ]:
# Last 20 filings using the IFRS Taxonomy (2024)
ifrs_filings = xbrl.report(
    endpoint="/report/search",
    parameters={
        "report_is_most_current": "true",
        "report_base_taxonomy": "IFRS 2024"
    },
    fields=["report.*"],
    sort={"report_accepted_timestamp": "desc"},
    limit=20,
    as_dataframe=True,
    print_query=True, # you can verify the query generated by pritnting it
)

ifrs_filings.head()

In [ ]:
# Search for facts with specific report ID
fact_response = xbrl.fact(  # you can use `xbrl.query()` as well.
    endpoint="/fact/search",
    parameters={
        "report_id": 207773,
        "member_is_base": "false"
    },
    fields=[
        "fact.numerical-value",
        "entity.cik",
        "entity.name",
        "report.filing-date",
        "concept.local-name",
        "fact.ultimus-index",
        "dimensions.count",
        "period.fiscal-year",
        "period.fiscal-period",
        "report.sic-code",
        "period.calendar-period",
        "dimension.is-base",
        "member.local-name",
    ],
    limit=10,
    as_dataframe=True
)

fact_response.head()

In [ ]:
# Get concept details for the concept Assets in the 2018 US GAAP Taxonomy
concept_reponse = xbrl.concept(  # you can use `xbrl.query()` as well.
    endpoint="/concept/Assets/search",
    parameters={
        "dts_id": 292503,
    },
    fields=[
        "concept.*",
        "label.text",
        "reference.*",
        "parts.local-name",
        "parts.part-value",
    ],
    as_dataframe=True,
)

concept_reponse.head()

The code below can replace the workflow found on:
https://hub.gesis.mybinder.org/user/xbrlus-xbrl-api-ipynb-yqnyffu2/notebooks/xbrl_us_api_fulldataresults.ipynb

As mentioned, the `XBRL` object manages authentication and token renewal. If your username is already saved, you do not need to provide your username and password again when initializing the object. and you can simply type:

```python
xbrl = XBRL()
```

In [ ]:
from xbrl_us import XBRL
import pandas as pd
from datetime import datetime

xbrl = XBRL() 

# Define query parameters
XBRL_Elements = [
    'CashCashEquivalentsAndShortTermInvestments',
    'EffectiveIncomeTaxRateReconciliationAtFederalStatutoryIncomeTaxRate',
    'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017Percent',
    'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017TransitionTaxOnAccumulatedForeignEarningsPercent',
    'EffectiveIncomeTaxRateReconciliationStateAndLocalIncomeTaxes',
    'EffectiveIncomeTaxRateReconciliationForeignIncomeTaxRateDifferential',
    'EffectiveIncomeTaxRateReconciliationTaxCredits',
    'EffectiveIncomeTaxRateReconciliationChangeInEnactedTaxRate',
    'EffectiveIncomeTaxRateReconciliationChangeInDeferredTaxAssetsValuationAllowance',
    'EffectiveIncomeTaxRateReconciliationShareBasedCompensationExcessTaxBenefitPercent',
    'EffectiveIncomeTaxRateReconciliationOtherAdjustments',
    'EffectiveIncomeTaxRateReconciliationOtherReconcilingItemsPercent',
    'EffectiveIncomeTaxRateContinuingOperations',
    'IncomeTaxReconciliationIncomeTaxExpenseBenefitAtFederalStatutoryIncomeTaxRate',
    'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017Amount',
    'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017TransitionTaxOnAccumulatedForeignEarningsAmount',
    'IncomeTaxReconciliationStateAndLocalIncomeTaxes',
    'IncomeTaxReconciliationForeignIncomeTaxRateDifferential',
    'IncomeTaxReconciliationTaxCredits',
    'IncomeTaxReconciliationOtherReconcilingItems'
]

# Record query start time
query_start = datetime.now()

# Execute the query with the xbrl_us library
response_df = xbrl.fact(
    endpoint="/fact/search",
    parameters={
        "concept_local_name": XBRL_Elements,
        "report_sic_code": [2834],
        "period_fiscal_year": [2021, 2020, 2019],
        "period_fiscal_period": ['Y'],
        "report_document_type": ["10-K", "10-K/A"],
        "fact_ultimus": "true",
    },
    fields=[
        "period_fiscal_year",
        "entity_name",
        "concept_local_name",
        "fact_numerical_value",
        "unit",
        "fact_decimals",
        "report_accession",
        "report_filing_date",
        "report_document_type",
        "report_sic_code"
    ],
    sort={
        "period_fiscal_year": "desc",
        "entity_name": "asc",
        "concept_local_name": "asc"
    },
    unique=True,  # Return only unique values
    as_dataframe=True,  # Return as pandas DataFrame
    print_query=True,  # Print the query for debugging
    limit="all", # Return all results
    async_mode=True,  # asynchronous mode can be used to speed up the query
    timeout=None,  # Set timeout to None for no limit
)

# Calculate query duration
query_end = datetime.now()
time_taken = query_end - query_start

# Display information about the query
print(f"\nQuery completed at {query_end.strftime('%c')}")
print(f"Retrieved {len(response_df)} rows in {time_taken}")

# Format and display results
pd.options.display.float_format = '{:,.2f}'.format
display_rows = 10  # Adjust as needed
response_df.head(display_rows)

# XBRL Using the Streamlit Web Interface

The XBRL-US Python package also offers a user-friendly web interface built with Streamlit. This graphical tool enables you to explore the data without having to write extensive code.

Key features include:
- Browsing various endpoints and discovering available parameters and fields.
- Viewing detailed definitions for each field.
- Executing queries on many endpoints directly from the interface.

This interactive approach is ideal if you prefer working with a GUI. You can launch the interface from a Jupyter Notebook by running:

```bash
!python -m xbrl_us
```

In [ ]:
!python -m xbrl_us